Origin: VicidominiLab/s2ISM, examples/Reconstruction_objects_increasing_radius.ipynb at 10bb5fc.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy import signal
import os

import brighteyes.imaging.simulation.PSF_sim as psf_simulator
import brighteyes.imaging.simulation.simulator as simulator
import brighteyes.imaging.analysis.Deconv_lib as deconv
import brighteyes.imaging.analysis.Tools_lib as tool
import brighteyes.imaging.analysis.Graph_lib as gra
from brighteyes.imaging.analysis import APR_lib as apr
import brighteyes.imaging.analysis.FocusISM_lib as fism

from brighteyes.imaging.s2ism import s2ism as s2


# PSF simulation

In [ ]:

b = np.finfo(float).eps

exPar = psf_simulator.simSettings()
exPar.na = 1.4   # numerical aperture
exPar.wl = 488  # wavelength [nm]
exPar.gamma = 45  # parameter describing the light polarization
exPar.beta = 90  # parameter describing the light polarization
exPar.n = 1.5
exPar.mask_sampl = 50

emPar = exPar.copy()
emPar.wl = 510


grid = psf_simulator.GridParameters()
grid.Nz = 2
grid.pxsizex = 40
grid.pxsizez = 550
grid.pxpitch = 75e3
grid.pxdim = 50e3
grid.N = 5
grid.Nx =101



psf_fin = psf_simulator.SPAD_PSF_3D(grid, exPar, emPar, stack='positive')[0]


In [ ]:
fig_1 = gra.ShowDataset(psf_fin[0])
fig_2 = gra.ShowDataset(psf_fin[1])

In [ ]:
N = 5
Nz = psf_fin.shape[0]

grid2 = psf_simulator.GridParameters()
grid2.Nz = 2
grid2.pxsizex = 40
grid2.pxsizez = 280
grid2.pxpitch = 75e3
grid2.pxdim = 50e3
grid2.N = 5
grid2.Nx =101

psf_fin_s2ism = psf_simulator.SPAD_PSF_3D(grid2, exPar, emPar, stack='positive')[0]


# Building phantoms

In [ ]:

def disk(nx, radius, offset = [0,0]):
    
    x = np.arange(-Nx//2, Nx//2, 1) + offset[0]
    y = np.arange(-Nx//2, Nx//2, 1) + offset[1]
    
    X, Y = np.meshgrid(x,y)
    
    r = np.sqrt(X**2 + Y**2)
    
    disk = np.where(r<=radius, 1, 0)
    
    return disk


In [ ]:
Nx = 401

radius = np.arange(5,60,5)

x_line = radius * grid.pxsizex / emPar.airy_unit

disks = np.empty((len(radius), 2, Nx, Nx))

offsets_focus = [(0,130), (0,-130), (130,0), (-130,0), (0,0)]

offsets_bkg  = [(130,130), (-130,130), (+130,-130), (-130,-130)]

disks_focus = np.zeros((len(radius), Nx, Nx))

disks_bkg = np.zeros((len(radius), Nx, Nx))

disks = np.empty((2,len(offsets_focus), Nx, Nx))



for i in range(len(radius)):
    for j in range(len(offsets_focus)):

        disks_focus[i] += disk(Nx, radius[i], offsets_focus[j])

for i in range(len(radius)):
     for j in range(len(offsets_bkg)):

        disks_bkg[i] += disk(Nx, radius[i], offsets_bkg[j])
    
disks = np.stack((disks_focus, disks_bkg), axis=0)


In [ ]:
disks = np.where(np.isnan(disks),0,disks)

In [ ]:
print(disks_focus.shape)
print(disks_bkg.shape)
disks = np.stack((disks_focus, disks_bkg), axis=0)
print(disks.shape)
print(psf_fin.shape)

Plotting of the phantom disks with smallest radius:

In [ ]:
show_radius = 0

fig1 = gra.ShowImg(disks[1,show_radius],grid.pxsizex*1e-3)
fig2 = gra.ShowImg(disks[0,show_radius],grid.pxsizex*1e-3)

# Forward model

In [ ]:
blur = np.zeros([len(radius),2, Nx, Nx, N**2])

data_ISM = np.zeros([len(radius), Nx, Nx, N**2])

sig = 90

for j in range(Nz):
    psf_fin[j, :, :, :] /= np.sum(psf_fin[j, :, :, :])
    
for l in range (len(radius)):
    for i in range(N**2):
        for j in range(Nz):
            blur[l,j, :, :, i] = signal.convolve(
                disks[j, l, :, :], psf_fin[j, :, :, i],mode='same')
            data_ISM[l,:, :, i] += blur[l,j, :, :, i]
            
        
data_non_projected = np.random.poisson(lam=np.abs(sig*blur), size=None)

data_noise= np.sum(data_non_projected, axis=1)


Here, we see the working principle of s$^2$ISM in action.

The images from the center of the detector array are dominated by the in-focus structure, while the images from the periphery of the detector array are dominated by the out-of-focus structure.

s$^2$ISM leverages the different modulation on the detector plane (the _fingerprint_) coordinates to separate the two different axial components.

In [ ]:
fig = gra.ShowDataset(data_noise[-1])

# Launching reconstructions

In [ ]:
data_apr = np.empty(data_noise.shape)

ups = 10
ref = data_apr[0].shape[-1]//2

for i in range(len(radius)):
    data_apr[i] = apr.APR(data_noise[i], ups, ref)[1]
    

In [ ]:
niter = 30

rec = np.empty([niter+1, len(radius), 2, Nx, Nx])

for i in range(len(radius)):

    rec[:,i], *_   = s2.max_likelihood_reconstruction(data_noise[i], psf_fin_s2ism, max_iter = niter, rep_to_save = 'all')


In [ ]:
from skimage import restoration

bkg_roll = np.empty([len(radius), Nx, Nx])

for i in range(len(radius)):
    bkg_roll[i] = restoration.rolling_ball(data_apr[i].sum(-1), radius = 6)

rec_roll_ball = np.squeeze(data_apr.sum(-1)) - bkg_roll
rec_roll_ball = np.where(rec_roll_ball<0,0,rec_roll_ball)

In [ ]:
rec_fism  = np.empty([len(radius), Nx, Nx])

calibr = psf_fin[0]

for i in range(len(radius)):

    rec_fism[i], *_ = fism.focusISM(data_noise[i], threshold = 1, calibration = calibr)
           

# Plotting results

Plotting the in-focus reconstruction of every background removal algorithm, for every radius (first two lines dedicated to the in-focus and out-of-focus ground truths respectively, for qualitative comparison purposes):

In [ ]:
gt = disks*sig

inv_iter = -1

titles = ['ground truth in-focus',
          'ground truth out-of-focus',
          'APR',
          'rolling ball',
          'focus ISM',
          's2ISM']
idx_focus = 0
idx_oof = 1

results =[gt[idx_focus], gt[idx_oof], data_apr.sum(-1), rec_roll_ball , rec_fism, rec[inv_iter, :, idx_focus] ]

fig, ax = plt.subplots(len(results),len(radius), sharex = True, sharey = True, figsize = (25,20))


for n in range(len(radius)):
    for m in range(len(results)):
    
        gra.ShowImg(results[m][n], grid.pxsizex*1e-3, fig = fig, ax = ax[m,n], clabel ='counts', cmap='magma')
        ax[m,n].axis('on')
        ax[m,n].set_xticks([])
        ax[m,n].set_yticks([])
        ax[m,0].set_ylabel(titles[m])
        ax[0,n].set_title(f'{x_line[n]:.2f} AU')
        

   
fig.tight_layout()

Detecting in-focus and out-of-focus patches valid for every reconstruction at every radius for SBR calculation and their plots:

In [ ]:
fig, ax = plt.subplots(1,2, sharex = 'row', sharey = 'row', figsize = (16,10))
gra.ShowImg(data_apr.sum(-1)[-1], grid.pxsizex*1e-3, fig = fig, ax = ax[0], clabel ='counts', cmap='magma')
rect_oof = patches.Rectangle((0, 0), 140, 140, linewidth=1, edgecolor='w', facecolor='none')
ax[0].add_patch(rect_oof)

gra.ShowImg(data_apr.sum(-1)[-1], grid.pxsizex*1e-3, fig = fig, ax = ax[1], clabel ='counts', cmap='magma')
rect_focus = patches.Rectangle((0, 140), 140, 265-140, linewidth=1, edgecolor='w', facecolor='none')
ax[1].add_patch(rect_focus)

In [ ]:
list_results_bkg =[data_apr.sum(-1)[:,:140,:140], rec_roll_ball[:,:140,:140], rec_fism[:,:140,:140] , rec[inv_iter,:,idx_focus][:,:140,:140]]

list_results_focus =[data_apr.sum(-1)[:,:140,140:265], rec_roll_ball[:,:140,140:265], rec_fism[:,:140,140:265] , rec[inv_iter,:,idx_focus][:,:140,140:265]]

opt_sect_ratio = np.empty((len(radius),len(list_results_bkg)))

# Plotting optical sectioning capabilities gain

In [ ]:
sbr = np.empty((len(radius),len(list_results_bkg)))

titles_graph = ['APR', 'rolling ball', 'focus ISM', 's2ISM']

for i in range(len(radius)):
    for j in range(len(list_results_bkg)):
        
        sbr[i,j] = list_results_focus[j][i].sum() / list_results_bkg[j][i].sum() - 1
        
        opt_sect_ratio[i,j] = sbr[i,j] / (1+np.abs(sbr[i,j]))
    
plt.figure()
for i in range(len(list_results_bkg)):
    plt.plot(x_line, opt_sect_ratio[:,i],label = titles_graph[i])
plt.xlabel('radius (AU)')
plt.ylabel('Normalized optical sectioning')
plt.legend()